<a href="https://colab.research.google.com/github/Maddox159-crypto/2026_code_online_competition/blob/main/pii_ff_4seed_ens_train_infer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PII Data Detection — DeBERTa-v3-base FULL_FIT 4시드 앙상블 (학습 + 추론)

**코드 공유용 노트북.** 최종 제출(private **0.96007** / public 0.96462)을 만든 과정을 한 노트북에 합쳤다. 실제 제출은 학습과 추론을 나눠서 했다.
- 학습: 시드 42는 Kaggle 커널 `pii-deberta-v2-fullfit`, 시드 1–3은 Colab A100
- 추론: `pii-deberta-v2-ff-ens` Version 2

| 단계 | 내용 |
| --- | --- |
| 데이터 | 대회 `train.json` 100% + 외부 데이터(nbroad, mpware 생성 에세이)를 epoch마다 30% 샘플링 |
| 모델 | `microsoft/deberta-v3-base`, 윈도우 1024 토큰(겹침 256), PII 없는 문서는 epoch마다 30% 샘플링 |
| 학습 | FULL_FIT 3에폭 × 시드 4개 (42, 1, 2, 3) |
| 추론 | 4개 모델의 단어별 확률 평균 → O 임계값 0.98 |
| 후처리 | drop_space, ws_fill, name_case, url_filter, name_prop |

설정(3에폭, thr 0.98, 후처리)은 train 10% 검증 분할에서 고른 값이다(v2 VALID F5 0.9686).

**입력**: 대회 데이터, `debarshichanda/debertav3base`, `nbroad/pii-dd-mistral-generated`, `mpware/pii-mixtral8x7b-generated-essays` · GPU ON, Internet OFF

In [ ]:
import os, gc, re, json, glob, math, time, random, pickle
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_cosine_schedule_with_warmup

DEBUG = os.environ.get("PII_DEBUG") == "1"          # 로컬 스모크 테스트용
INPUT_ROOT = os.environ.get("PII_INPUT", "/kaggle/input")
WORK_DIR = os.environ.get("PII_WORK", "/kaggle/working")

SEED = 42
SEEDS = [42, 1, 2, 3]   # 앙상블할 학습 시드
TRAIN_SEED = SEEDS[0]    # 학습 랜덤성(초기화·샘플링·셔플): 아래 시드 루프에서 바꿈
MAX_LEN = 1024          # 윈도우 길이(특수토큰 포함)
STRIDE = 256            # 윈도우 겹침
EPOCHS = 3
LR = 2.5e-5
WARMUP = 0.1
TRAIN_BS = 2
INFER_BS = 8
NEG_RATIO = 0.3         # PII 없는 문서 샘플링 비율(epoch마다 다시 뽑음)
# ---- 외부 데이터 (학습에만 사용) ----
USE_EXT = True
EXT_FILES = ["mixtral-8x7b-v1.json", "mpware_mixtral8x7b_v1.1-no-i-username.json"]
EXT_RATIO = 0.3         # epoch마다 외부 문서 샘플링 비율
# ---- FULL_FIT: v2 검증(VALID F5 0.9686)에서 고른 설정 고정 → train 100%로 학습 ----
FIXED_EP, FIXED_THR = 3, 0.98
FIXED_PP = dict(drop_space=1, ws_fill=1, name_case=1, url_filter=1, name_prop=1)
if DEBUG:
    MAX_LEN, STRIDE, EPOCHS, TRAIN_BS, INFER_BS = 128, 32, 2, 4, 8
    FIXED_EP = 1

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(TRAIN_SEED)

def find_comp_file(name, required=True):
    c = [p for p in glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True)]
    c = sorted(c, key=lambda p: ("pii-detection" not in p, len(p)))
    if not c:
        assert not required, f"{name} 를 찾을 수 없습니다. 대회 데이터를 Input에 추가하세요."
        return None
    return c[0]

def find_backbone():
    cands = []
    for cfg in glob.glob(f"{INPUT_ROOT}/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        files = os.listdir(d)
        has_w = any(f.endswith((".bin", ".safetensors")) for f in files)
        has_tok = any(f in files for f in ("spm.model", "tokenizer.json"))
        if has_w and has_tok:
            cands.append(d)
    assert cands, "DeBERTa-v3-base 가중치를 찾을 수 없음"
    cands.sort(key=lambda d: ("deberta" not in d.lower(), "v3" not in d.lower(), len(d)))
    return cands[0]

TRAIN_PATH = find_comp_file("train.json")
TEST_PATH = find_comp_file("test.json")
BACKBONE = os.environ.get("PII_BACKBONE") or find_backbone()
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device", device,
      torch.cuda.get_device_name(0) if device == "cuda" else "")
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)
print("BACKBONE  :", BACKBONE)
print(f"SEEDS={SEEDS} USE_EXT={USE_EXT} EXT_RATIO={EXT_RATIO} EPOCHS={FIXED_EP} THR={FIXED_THR}")
if not DEBUG and device != "cuda":
    print(" GPU가 꺼져 있음.")

LABELS = ["B-EMAIL", "B-ID_NUM", "B-NAME_STUDENT", "B-PHONE_NUM", "B-STREET_ADDRESS", "B-URL_PERSONAL",
          "B-USERNAME", "I-ID_NUM", "I-NAME_STUDENT", "I-PHONE_NUM", "I-STREET_ADDRESS", "I-URL_PERSONAL", "O"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
O_ID = label2id["O"]
NUM_LABELS = len(LABELS)

with open(TRAIN_PATH, encoding="utf-8") as f:
    train_data = json.load(f)
with open(TEST_PATH, encoding="utf-8") as f:
    test_data = json.load(f)
print("train docs:", len(train_data), "| test docs:", len(test_data))

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
assert tokenizer.is_fast, "fast tokenizer 필요 (offset mapping)"
CLS_ID, SEP_ID = tokenizer.cls_token_id, tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

def encode_doc(doc, with_labels=True):
    """spaCy 토큰 → 문자열 재구성 → 서브워드 토큰화. 서브워드별로 원래 토큰 index 매핑."""
    text, tmap = [], []
    for i, (t, ws) in enumerate(zip(doc["tokens"], doc["trailing_whitespace"])):
        text.append(t); tmap.extend([i] * len(t))
        if ws:
            text.append(" "); tmap.append(-1)
    text = "".join(text)
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    ids = enc["input_ids"]
    # 서브워드 하나가 여러 spaCy 토큰에 걸칠 수 있으므로 (서브워드, 토큰) 쌍을 모두 기록
    pair_sub, pair_word, first_word = [], [], []
    for k, (s, e) in enumerate(enc["offset_mapping"]):
        ws = []
        for c in range(s, min(e, len(tmap))):
            w = tmap[c]
            if w != -1 and (not ws or ws[-1] != w):
                ws.append(w)
        first_word.append(ws[0] if ws else -1)
        for w in ws:
            pair_sub.append(k); pair_word.append(w)
    out = {"ids": ids, "sub2word": np.array(first_word, dtype=np.int64),
           "pair_sub": np.array(pair_sub, dtype=np.int64), "pair_word": np.array(pair_word, dtype=np.int64),
           "n_words": len(doc["tokens"]), "text": text, "offsets": enc["offset_mapping"]}
    if with_labels:
        wl = np.array([label2id[l] for l in doc["labels"]])
        lab = np.full(len(ids), -100, dtype=np.int64)
        for k, w in zip(pair_sub, pair_word):
            if lab[k] == -100 or (lab[k] == O_ID and wl[w] != O_ID):
                lab[k] = wl[w]          # 걸친 토큰 중 첫 PII 라벨 우선
        out["labels"] = lab
    return out

def make_windows(n):
    """길이 n 시퀀스를 (MAX_LEN-2) 크기, STRIDE 겹침으로 자르는 시작 위치들"""
    L = MAX_LEN - 2
    if n <= L:
        return [0]
    step = L - STRIDE
    starts = list(range(0, n - L, step)) + [n - L]
    return sorted(set(starts))

In [ ]:
# ---- 외부 데이터 (학습에만 사용) ----
ext_data = []
if USE_EXT:
    for fn in EXT_FILES:
        p = find_comp_file(fn, required=False)
        if p is None:
            print("외부 데이터 없음 (건너뜀):", fn); continue
        with open(p, encoding="utf-8") as f:
            ext_data += json.load(f)
    ext_data = [d for d in ext_data if set(d["labels"]) <= set(LABELS)]
if DEBUG:
    train_data, ext_data = train_data[:400], ext_data[:40]
print("external docs:", len(ext_data))

In [ ]:
# ---- FULL_FIT: train 100% 사용 ----
# 제출 모델과 같은 문서 순서(검증 분할의 train → valid 순)로 이어 붙임 → 시드별 샘플링·셔플 재현
rng = np.random.RandomState(SEED)
has_pii = np.array([any(l != "O" for l in d["labels"]) for d in train_data])
val_idx = []
for flag in (True, False):
    idx = np.where(has_pii == flag)[0]
    rng.shuffle(idx)
    val_idx += list(idx[: int(len(idx) * 0.10)])
val_set = set(val_idx)
tr_docs = [d for i, d in enumerate(train_data) if i not in val_set] + [d for i, d in enumerate(train_data) if i in val_set]
print(f"train docs (100%): {len(tr_docs)}")
tr_enc = [encode_doc(d) for d in tqdm(tr_docs, desc="encode train")]

# 정렬 검증: 서브워드 라벨 → 토큰 라벨로 되돌렸을 때 PII 토큰이 빠짐없이 커버되는지
lost = total = 0; lost_ex = []
for d, e in zip(tr_docs, tr_enc):
    covered = set(e["pair_word"].tolist())
    for i, l in enumerate(d["labels"]):
        if l != "O":
            total += 1
            if i not in covered:
                lost += 1; lost_ex.append((l, d["tokens"][i]))
print(f"alignment check: PII tokens {total}, not covered by any subword {lost} ({lost/max(total,1):.3%})")
from collections import Counter
print("  uncovered examples:", Counter((l, repr(t)) for l, t in lost_ex).most_common(10))
e0 = tr_enc[0]
print("  offset sanity:", [(tokenizer.convert_ids_to_tokens(e0["ids"][k]), e0["text"][s:e]) for k, (s, e) in enumerate(e0["offsets"][:12])])
for e in tr_enc:
    e.pop("text"); e.pop("offsets")

def doc_windows(e):
    out = []
    for s in make_windows(len(e["ids"])):
        ids = [CLS_ID] + e["ids"][s:s + MAX_LEN - 2] + [SEP_ID]
        lab = [-100] + list(e["labels"][s:s + MAX_LEN - 2]) + [-100]
        out.append((ids, lab))
    return out

def split_windows(encs):
    """PII 있는 문서의 윈도우(항상 사용) / PII 없는 문서별 윈도우(epoch마다 샘플링)"""
    pos, neg = [], []
    for e in encs:
        w = doc_windows(e)
        if ((e["labels"] >= 0) & (e["labels"] != O_ID)).any():
            pos += w
        else:
            neg.append(w)
    return pos, neg

pos_windows, neg_doc_windows = split_windows(tr_enc)
print("positive-doc windows:", len(pos_windows), "| negative docs:", len(neg_doc_windows))

ext_doc_windows = []
for d in tqdm(ext_data, desc="encode external"):
    ext_doc_windows.append(doc_windows(encode_doc(d)))
print("external docs:", len(ext_doc_windows), "| windows:", sum(map(len, ext_doc_windows)))

def collate(batch):
    m = max(len(x[0]) for x in batch)
    ids = torch.full((len(batch), m), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(batch), m), dtype=torch.long)
    lab = torch.full((len(batch), m), -100, dtype=torch.long)
    for i, (a, b) in enumerate(batch):
        ids[i, :len(a)] = torch.tensor(a); att[i, :len(a)] = 1; lab[i, :len(b)] = torch.tensor(b)
    return ids, att, lab

In [ ]:
# ---- 공식 대회 지표: (document, token, BIO label) 일치 기준 micro F-beta(β=5) ----
def compute_f5(pred_rows, gt_rows, beta=5):
    preds = set(pred_rows); gts = set(gt_rows)
    tp = len(preds & gts); fp = len(preds - gts); fn = len(gts - preds)
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = (1 + beta**2) * p * r / (beta**2 * p + r) if (p + r) else 0.0
    per = defaultdict(lambda: [0, 0, 0])
    for x in preds & gts: per[x[2][2:]][0] += 1
    for x in preds - gts: per[x[2][2:]][1] += 1
    for x in gts - preds: per[x[2][2:]][2] += 1
    return f, p, r, dict(per)


@torch.no_grad()
def predict_word_probs(model, encs):
    """문서별 [n_words, NUM_LABELS] 확률. 윈도우 겹침 평균 → 서브워드 평균으로 단어 확률."""
    model.eval()
    items = []
    for di, e in enumerate(encs):
        for s in make_windows(len(e["ids"])):
            items.append((di, s, [CLS_ID] + e["ids"][s:s + MAX_LEN - 2] + [SEP_ID]))
    items.sort(key=lambda x: len(x[2]))
    sub_sum = [np.zeros((len(e["ids"]), NUM_LABELS), np.float32) for e in encs]
    sub_cnt = [np.zeros((len(e["ids"]), 1), np.float32) for e in encs]
    for b in tqdm(range(0, len(items), INFER_BS), desc="predict", leave=False):
        chunk = items[b:b + INFER_BS]
        ids, att, _ = collate([(x[2], []) for x in chunk])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(input_ids=ids.to(device), attention_mask=att.to(device)).logits
        probs = torch.softmax(logits.float(), -1).cpu().numpy()
        for (di, s, seq), pr in zip(chunk, probs):
            n = len(seq) - 2
            sub_sum[di][s:s + n] += pr[1:1 + n]
            sub_cnt[di][s:s + n] += 1
    return [sub_to_word(e, ss / np.maximum(sc, 1)) for e, ss, sc in zip(encs, sub_sum, sub_cnt)]

def sub_to_word(e, sp):
    """서브워드 확률 [n_sub, C] → spaCy 토큰 확률 [n_words, C] (토큰을 덮는 서브워드 평균)"""
    wp = np.zeros((e["n_words"], NUM_LABELS), np.float32); wc = np.zeros((e["n_words"], 1), np.float32)
    np.add.at(wp, e["pair_word"], sp[e["pair_sub"]]); np.add.at(wc, e["pair_word"], 1)
    wp = wp / np.maximum(wc, 1)
    wp[wc[:, 0] == 0, O_ID] = 1.0   # 서브워드가 없는 토큰(공백 등)은 O
    return wp

# ---- 후처리 ----
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE = re.compile(r"(\(\d{3}\)\s?\d{3}[\-\.\s]\d{4}|\d{3}[\-\.]\d{3}[\-\.]\d{4})")
NAME_OK = re.compile(r"[A-Z][A-Za-z'\-\.]*")
URL_OK = re.compile(r".*(\.[A-Za-z]{2,}|/).*")      # 점+도메인 또는 슬래시가 있어야 URL로 인정
TYPES = sorted({l[2:] for l in LABELS if l != "O"})

def thr_vec(thr):
    """thr: float(전체 공통) 또는 {"_": 기본값, TYPE: 값} → 라벨별 O-임계값 배열"""
    if not isinstance(thr, dict):
        thr = {"_": thr}
    return np.array([thr.get(l[2:], thr["_"]) if l != "O" else 0.0 for l in LABELS], np.float32)

def decode(docs, word_probs, thr, pp):
    rows = []
    tv = thr_vec(thr)
    for d, wp in zip(docs, word_probs):
        toks = d["tokens"]
        o = wp[:, O_ID]
        nonO = wp.copy(); nonO[:, O_ID] = -1
        cand = nonO.argmax(1)
        pred = np.where(o < tv[cand], cand, O_ID)
        labs = [id2label[int(p)] for p in pred]
        for i, t in enumerate(toks):
            l = labs[i]
            if l == "O":
                continue
            typ = l[2:]
            if pp.get("drop_space") and t.strip() == "" and typ != "STREET_ADDRESS":
                labs[i] = "O"; continue
            if pp.get("name_case") and typ == "NAME_STUDENT" and not NAME_OK.fullmatch(t):
                labs[i] = "O"; continue
            if pp.get("url_filter") and typ == "URL_PERSONAL" and not URL_OK.fullmatch(t):
                labs[i] = "O"; continue
        if pp.get("name_prop"):
            # 문서 안에서 이름으로 예측된 토큰과 똑같은 토큰이 다른 곳에 O로 남아 있으면 이름으로 채움
            names = {t for t, l in zip(toks, labs) if l[2:] == "NAME_STUDENT" and NAME_OK.fullmatch(t) and len(t) > 1}
            for i, t in enumerate(toks):
                if labs[i] == "O" and t in names:
                    labs[i] = "I-NAME_STUDENT" if i > 0 and labs[i - 1][2:] == "NAME_STUDENT" else "B-NAME_STUDENT"
        if pp.get("ws_fill"):
            # 공백/개행 토큰이 같은 타입 엔티티 사이에 끼어 있으면 I-타입으로 채움 (주소 줄바꿈 등)
            for i, t in enumerate(toks):
                if t.strip() == "" and labs[i] == "O" and 0 < i < len(toks) - 1:
                    a, b = labs[i - 1], labs[i + 1]
                    if a != "O" and b.startswith("I-") and a[2:] == b[2:]:
                        labs[i] = b
        if pp.get("regex"):
            for i, t in enumerate(toks):
                if EMAIL_RE.fullmatch(t):
                    labs[i] = "B-EMAIL"
        if pp.get("bio_fix"):
            prev = "O"
            for i, l in enumerate(labs):
                if l != "O":
                    typ = l[2:]
                    if l.startswith("I-") and prev[2:] != typ:
                        # 앞 토큰이 공백이고 그 앞이 같은 타입이면 I 유지
                        j = i - 1
                        while j >= 0 and toks[j].strip() == "" and labs[j] == "O":
                            j -= 1
                        if not (j >= 0 and labs[j][2:] == typ and typ == "STREET_ADDRESS"):
                            labs[i] = "B-" + typ if ("B-" + typ) in label2id else l
                    elif l.startswith("B-") and typ in ("STREET_ADDRESS", "PHONE_NUM", "URL_PERSONAL") and prev[2:] == typ:
                        labs[i] = "I-" + typ if ("I-" + typ) in label2id else l
                prev = labs[i] if toks[i].strip() != "" or labs[i] != "O" else prev
        for i, l in enumerate(labs):
            if l != "O":
                rows.append((d["document"], i, l))
    return rows

In [ ]:
# ---- 학습 ----
def epoch_windows(ep, pos_w, neg_w):
    r = random.Random(TRAIN_SEED + ep)
    w = list(pos_w)
    for pool, ratio in ((neg_w, NEG_RATIO), (ext_doc_windows, EXT_RATIO)):
        for doc in r.sample(pool, int(len(pool) * ratio)):
            w += doc
    r.shuffle(w)
    return w

def train_model(pos_w, neg_w, epochs, on_epoch_end=None):
    seed_everything(TRAIN_SEED)
    model = AutoModelForTokenClassification.from_pretrained(
        BACKBONE, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
    model = model.float().to(device)   # 가중치가 fp16으로 로드되는 경우 대비 (fp32 마스터 가중치 + AMP)
    steps_per_epoch = math.ceil(len(epoch_windows(0, pos_w, neg_w)) / TRAIN_BS)
    total_steps = steps_per_epoch * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = get_cosine_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for ep in range(epochs):
        model.train()
        wins = epoch_windows(ep, pos_w, neg_w)
        t0 = time.time(); run_loss = 0.0
        pbar = tqdm(range(0, len(wins), TRAIN_BS), desc=f"epoch {ep+1}/{epochs}")
        for k, b in enumerate(pbar):
            ids, att, lab = collate(wins[b:b + TRAIN_BS])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=ids.to(device), attention_mask=att.to(device)).logits
            loss = torch.nn.functional.cross_entropy(logits.float().view(-1, NUM_LABELS), lab.to(device).view(-1), ignore_index=-100)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            run_loss += loss.item()
            if k % 50 == 0:
                pbar.set_postfix(loss=f"{run_loss/(k+1):.4f}")
        print(f"epoch {ep+1} train loss {run_loss/len(pbar):.4f} ({(time.time()-t0)/60:.1f} min, {len(wins)} windows)")
        if on_epoch_end:
            on_epoch_end(model, ep)
    return model

In [ ]:
# ---- 시드별 FULL_FIT 학습 → 모델 저장 (시드당 T4 약 55분 / A100 약 13분) ----
MODEL_DIRS = []
for s in SEEDS:
    TRAIN_SEED = s                     # epoch_windows / train_model 이 읽는 전역값
    print(f"===== seed {s} =====")
    model = train_model(pos_windows, neg_doc_windows, FIXED_EP)
    md = os.path.join(WORK_DIR, f"pii_deberta_f{s}")
    model.save_pretrained(md); tokenizer.save_pretrained(md)
    MODEL_DIRS.append(md)
    del model; gc.collect(); torch.cuda.empty_cache()
    print("saved ->", md)

In [ ]:
# ---- 앙상블 추론: 모델별 단어 확률 평균 → thr 0.98 + 후처리 5종으로 디코딩 ----
te_enc = [encode_doc(d, with_labels=False) for d in tqdm(test_data, desc="encode test")]
te_probs = None
for md in MODEL_DIRS:
    model = AutoModelForTokenClassification.from_pretrained(md).float().to(device)
    if use_amp:
        model = model.half()
    p = predict_word_probs(model, te_enc)
    te_probs = p if te_probs is None else [a + b for a, b in zip(te_probs, p)]
    del model; gc.collect(); torch.cuda.empty_cache()
    print("done:", md)
te_probs = [a / len(MODEL_DIRS) for a in te_probs]
rows = decode(test_data, te_probs, FIXED_THR, FIXED_PP)
sub = pd.DataFrame(rows, columns=["document", "token", "label"])
sub.insert(0, "row_id", range(len(sub)))
sub.to_csv(os.path.join(WORK_DIR, "submission.csv"), index=False)
print("submission rows:", len(sub))
print(sub.label.value_counts())
sub.head(20)